# SENTINEL-WM - Complete Pipeline (one notebook, end to end)

Runs **every phase** on the CIC-IDS-2017 data in `data/`:

`preprocess -> state windows -> sequences -> baselines -> world model ->
neural zoo -> graph windows -> GAT -> benchmark -> forward simulation -> explainability`

Each phase has its own deep-dive notebook (`01`..`05`); this one is the
quick tour. Small epoch budgets so it finishes in a few minutes on a GPU.
For the full research run use `python -m sentinel_wm.research all`.

In [1]:
# bootstrap: make the sentinel_wm package importable without pip install
import sys, pathlib
_root = pathlib.Path.cwd()
_root = _root if (_root / 'sentinel_wm').is_dir() else _root.parent
if str(_root) not in sys.path: sys.path.insert(0, str(_root))

In [2]:
import json, numpy as np, pandas as pd, matplotlib.pyplot as plt
from sentinel_wm import config as C
device = C.resolve_device('auto'); print('device:', device)
print('raw CSVs:', [p.split(chr(92))[-1].split('/')[-1] for p in C.RAW_FLOW_CSVS])

device: cuda
raw CSVs: ['unified_AllDays_labeled.csv']


## Phase 1-2c - data -> state windows -> sequences

In [3]:
from sentinel_wm import preprocessing, state_windows, sequences
flows = preprocessing.load_and_clean(verbose=True)
sw = state_windows.build_state_windows(flows, verbose=True)
seq = sequences.build_sequences(sw, verbose=True)

[load] unified_AllDays_labeled.csv (2.37 GB)


  cleaned 2,829,609 -> 2,829,609 rows | 97 numeric model cols | 5,734 inf/NaN cells zero-filled
  family distribution:
BENIGN                      2271975
    DoS Hulk                     231073
    PortScan                     158927
    DDoS                         128027
    DoS GoldenEye                 10293
    FTP-Patator                    7938
    SSH-Patator                    5897
    DoS slowloris                  5794
    DoS Slowhttptest               5492
    Bot                            1966
    Web Attack Brute Force         1507
    Web Attack XSS                  652
    Infiltration                     36
    Web Attack SQL Injection         21
    Heartbleed                       11


[save] C:\Users\chall\OneDrive\Desktop\SIH\artifacts\clean_flows.parquet  (2,829,609 rows, 130 cols)
[days] ['Friday', 'Monday', 'Thursday', 'Tuesday', 'Wednesday']
[windows] W=10s stride=10s pre_attack_span=3


[windows] 14,644 state windows -> C:\Users\chall\OneDrive\Desktop\SIH\artifacts\state_windows.parquet
[windows] progression_state:
NORMAL          11829
    ACTIVE           1929
    PRE_ATTACK        583
    ONSET             242
    CONTINUATION       61
[windows] y_attack balance: 0.152 positive
[windows] ATT&CK phase x confidence:
attck_phase        attck_confidence
    Command & Control  Low                   405
                       Medium                 79
    Impact             High                  346
                       Medium                151
    Initial Access     Low                   729
                       Medium                391
    Lateral Movement   Low                    34
    None               High                11829
    Reconnaissance     High                   48
                       Low                   583
                       Medium                 49
[split] mode=block  days=['Friday', 'Monday', 'Thursday', 'Tuesday', 'Wednesday']


[seq] train:   8902 seqs | y_atk(any k)+ = 0.203 | k1+ = 0.141
[seq] val  :   2732 seqs | y_atk(any k)+ = 0.227 | k1+ = 0.189
[seq] test :   2602 seqs | y_atk(any k)+ = 0.225 | k1+ = 0.157
[seq] X shape (14236, 10, 41) -> C:\Users\chall\OneDrive\Desktop\SIH\artifacts\sequences.npz
[seq] scaler -> C:\Users\chall\OneDrive\Desktop\SIH\artifacts\state_scaler.pkl


## Phase 3 - classical baseline zoo
Logistic Regression, Random Forest, Extra Trees, HistGB, MLP, LinearSVC,
kNN, GaussianNB, XGBoost, LightGBM - each on the current window **and** on
the flattened L-window sequence (`__seq`), one classifier per horizon.

In [4]:
from sentinel_wm import baselines
_ = baselines.run_baselines(verbose=True)

[baseline] models=['logistic_regression', 'random_forest', 'extra_trees', 'hist_gradient_boosting', 'mlp_sklearn', 'linear_svc', 'knn', 'gaussian_nb', 'xgboost', 'lightgbm']
[baseline] xgboost=True lightgbm=True | train/val/test = 8902/2732/2602  K=6


[logistic_regression       ] F1=0.377 P=0.601 R=0.275 FPR=0.053 AUROC=0.781 (n=2602, pos=585)  MLT=10s det=0.14 (7.51s)


C:\Users\chall\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\linear_model\_logistic.py:814: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


C:\Users\chall\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\linear_model\_logistic.py:814: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


C:\Users\chall\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\linear_model\_logistic.py:814: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


...

[linear_svc                ] F1=0.379 P=0.646 R=0.268 FPR=0.043 AUROC=0.781 (n=2602, pos=585)  MLT=10s det=0.12 (16.82s)


[knn                       ] F1=0.430 P=0.764 R=0.299 FPR=0.027 AUROC=0.781 (n=2602, pos=585)  MLT=10s det=0.06 (6.52s)
[gaussian_nb               ] F1=0.382 P=0.435 R=0.340 FPR=0.128 AUROC=0.719 (n=2602, pos=585)  MLT=10s det=0.18 (0.06s)


[xgboost                   ] F1=0.601 P=0.783 R=0.487 FPR=0.039 AUROC=0.869 (n=2602, pos=585)  MLT=10s det=0.14 (5.69s)


[xgboost__seq              ] F1=0.678 P=0.794 R=0.591 FPR=0.045 AUROC=0.906 (n=2602, pos=585)  MLT=10s det=0.17 (74.11s)
[baseline] lightgbm/window FAILED: exception: access violation reading 0x0000000000000000


[baseline] lightgbm/sequence FAILED: exception: access violation reading 0x0000000000000000
[baseline] 15 model variants -> C:\Users\chall\OneDrive\Desktop\SIH\artifacts\baselines\baseline_metrics.json
[baseline] fitted estimators -> C:\Users\chall\OneDrive\Desktop\SIH\research\models\classical


## Phase 4 - the world model (Temporal Transformer + probabilistic STN)

In [5]:
from sentinel_wm import train
import argparse
ns = argparse.Namespace(test=False, epochs=15, batch_size=None, lr=None, device=device)
_ = train.train(C.CONFIG, ns)

[train] device=cuda
[train] features=41 L=10 K=6 | train=8902 val=2732 test=2602


[train] attack_pos_weight=6.06 prog_w=[0.17 0.78 1.23 0.45 2.37]


  ep001  2.5s L=5.0616 (atk 2.326 prog 2.755 next 0.782) | val F1=0.185 AUROC=0.759 progAcc=0.785 MLT=10s


  ep002  1.2s L=4.3909 (atk 2.010 prog 2.450 next 0.607) | val F1=0.407 AUROC=0.830 progAcc=0.810 MLT=10s


  ep003  1.2s L=3.7354 (atk 1.646 prog 2.255 next 0.556) | val F1=0.100 AUROC=0.769 progAcc=0.701 MLT=10s


...

  ep020  1.2s L=1.4916 (atk 0.537 prog 1.236 next 0.271) | val F1=0.675 AUROC=0.884 progAcc=0.778 MLT=10s


  ep021  1.2s L=1.5917 (atk 0.589 prog 1.279 next 0.273) | val F1=0.687 AUROC=0.889 progAcc=0.818 MLT=10s


  ep022  1.2s L=1.4297 (atk 0.510 prog 1.197 next 0.264) | val F1=0.679 AUROC=0.889 progAcc=0.826 MLT=10s


  ep023  1.2s L=1.3525 (atk 0.471 prog 1.161 next 0.263) | val F1=0.739 AUROC=0.888 progAcc=0.847 MLT=10s


  ep024  1.4s L=1.3297 (atk 0.462 prog 1.140 next 0.266) | val F1=0.677 AUROC=0.885 progAcc=0.831 MLT=10s
[train] early stop at epoch 24

================  WORLD MODEL - TEST  ================
any-horizon            F1=0.622 P=0.631 R=0.614 FPR=0.104 AUROC=0.854 (n=2602, pos=585)
  k  horizon     F1   prec    rec    FPR  AUROC  Brier
  1     10s   0.670  0.592  0.770  0.098  0.912  0.081
  2     20s   0.658  0.582  0.757  0.101  0.909  0.083
  3     30s   0.663  0.590  0.756  0.098  0.904  0.084
  4     40s   0.628  0.548  0.735  0.114  0.893  0.091
  5     50s   0.626  0.548  0.730  0.113  0.888  0.091
  6     60s   0.612  0.528  0.727  0.121  0.882  0.097
progression acc (all k) : 0.854   k=1 : 0.865
Mean Lead Time          : 10.0 s  (median 10s, max 10s)
episodes warned         : 68/313 (0.22)   false-alarm rate 0.036
Brier(k1) 0.081   ECE(k1) 0.059   alert threshold 0.43

[train] checkpoint -> C:\Users\chall\OneDrive\Desktop\SIH\artifacts\world_model.pt


## Phase 4b - neural sequence baselines (MLP / LSTM / GRU / TCN)

In [6]:
from sentinel_wm.nn_common import run_nn_zoo
_ = run_nn_zoo(kinds=['mlp','lstm','gru','tcn'], epochs=12, device=device, verbose=True)


=== NN baseline: mlp ===


  [mlp] ep001  0.4s L=1.6717 valF1=0.250 AUROC=0.841 prog=0.756


  [mlp] ep002  0.4s L=1.3089 valF1=0.587 AUROC=0.859 prog=0.818


  [mlp] ep003  0.4s L=1.1290 valF1=0.612 AUROC=0.869 prog=0.857


  [mlp] ep004  0.4s L=1.0136 valF1=0.644 AUROC=0.869 prog=0.857


...

  [tcn] ep008  0.5s L=0.8058 valF1=0.642 AUROC=0.878 prog=0.837


  [tcn] ep009  0.5s L=0.7691 valF1=0.641 AUROC=0.876 prog=0.838


  [tcn] ep010  0.5s L=0.7425 valF1=0.636 AUROC=0.869 prog=0.827


  [tcn] ep011  0.5s L=0.7316 valF1=0.636 AUROC=0.869 prog=0.829


  [tcn] ep012  0.6s L=0.7278 valF1=0.635 AUROC=0.869 prog=0.830
[tcn] TEST F1=0.547 AUROC=0.830 MLT=10s prog=0.823 (168k params)


## Phase 4c - Graph Attention Network
Per-window host graphs (who talked to whom) -> from-scratch GAT -> GRU over time.

In [7]:
from sentinel_wm.graph_windows import build_graph_windows
from sentinel_wm.gat import run_gat
build_graph_windows(verbose=True)
_ = run_gat(epochs=12, device=device, verbose=True)

  graph windows 2000/14644


  graph windows 4000/14644


  graph windows 6000/14644


  graph windows 8000/14644


  graph windows 10000/14644


...

  [gat] ep009  4.4s L=1.0819 valF1=0.794 AUROC=0.904 prog=0.862


  [gat] ep010  4.3s L=1.0292 valF1=0.798 AUROC=0.913 prog=0.900


  [gat] ep011  4.7s L=1.0094 valF1=0.798 AUROC=0.913 prog=0.921


  [gat] ep012  4.3s L=0.9965 valF1=0.792 AUROC=0.910 prog=0.916


[gat] TEST F1=0.699 AUROC=0.883 MLT=10s prog=0.863 (247k params)


## Phase 5 - unified benchmark (every model, same test anchors)

In [8]:
from sentinel_wm import benchmark
from sentinel_wm.registry import build_registry
data = benchmark.run_benchmark(device=device, verbose=False)
build_registry(verbose=False)
bf = pd.read_csv('../../runs/benchmarks/benchmark_full.csv')
bf[['model','family','f1','auroc','mean_lead_time_s','detection_rate','progression_acc']].round(3)

[bench] figures -> C:\Users\chall\OneDrive\Desktop\SIH\research\figures


,model,family,f1,auroc,mean_lead_time_s,detection_rate,progression_acc
0,persistence,reference,0.770,0.820,0.0,0.000,NaN
1,gat,graph,0.699,0.883,10.0,0.115,0.863
2,xgboost__seq,classical,0.678,0.906,10.0,0.166,NaN
3,random_forest__seq,classical,0.648,0.896,10.0,0.141,NaN
4,hist_gradient_boosting__seq,classical,0.643,0.912,10.0,0.163,NaN
5,SENTINEL-WM,world_model,0.622,0.854,10.0,0.217,0.854
6,mlp_sklearn__seq,classical,0.601,0.813,10.0,0.096,NaN
7,xgboost,classical,0.601,0.869,10.0,0.144,NaN
8,extra_trees,classical,0.590,0.850,10.0,0.157,NaN
9,logistic_regression__seq,classical,0.576,0.854,10.0,0.089,NaN


In [9]:
ph = pd.read_csv('../../runs/benchmarks/per_horizon_f1.csv').set_index('model')
ax = ph.T.plot(figsize=(9,5), marker='o'); ax.set_ylabel('F1'); ax.set_xlabel('horizon')
ax.set_title('Forecast-horizon F1 - every model'); ax.legend(fontsize=7, ncol=2); plt.tight_layout()

## Phase 6 - forward simulation + explainability

In [10]:
from sentinel_wm import forward_sim, explain
res = forward_sim.simulate_split('test', limit=120, device=device, with_explain=True)
atk = sorted([r for r in res if r['meta'].get('y_now')==1], key=lambda r:-r['max_attack_prob'])
r = atk[0]
for h in r['horizon']:
    a=h['attck']; print(f"+{h['horizon_seconds']:>3}s P={h['attack_prob']:.2f} "
          f"{h['progression_state']:<12} {a['kill_chain_phase']:<20} {a['confidence']}")

C:\Users\chall\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


+ 10s P=0.93 ACTIVE       Command & Control    Low
+ 20s P=0.89 ACTIVE       Command & Control    Low
+ 30s P=0.83 ACTIVE       Command & Control    Low
+ 40s P=0.78 ACTIVE       Command & Control    Low
+ 50s P=0.68 ACTIVE       Command & Control    Low
+ 60s P=0.63 NORMAL       None                 Low


In [11]:
full = explain.run_all(device)
tf = full['world_model_shap']['top_features'][:10]
plt.barh([t['feature'] for t in tf][::-1], [t['mean_abs_shap'] for t in tf][::-1])
plt.title(full['world_model_shap']['method']); plt.tight_layout()

  0%|          | 0/64 [00:00<?, ?it/s]

  2%|▏         | 1/64 [00:00<00:30,  2.07it/s]

  3%|▎         | 2/64 [00:00<00:17,  3.45it/s]

  5%|▍         | 3/64 [00:00<00:13,  4.39it/s]

  6%|▋         | 4/64 [00:00<00:12,  4.99it/s]

...

 97%|█████████▋| 62/64 [00:09<00:00,  6.75it/s]

 98%|█████████▊| 63/64 [00:09<00:00,  6.74it/s]

100%|██████████| 64/64 [00:09<00:00,  6.74it/s]

100%|██████████| 64/64 [00:09<00:00,  6.46it/s]

[explain] shap_installed=True
[explain] world-model top features:
    payload_kurt_mean          0.0774
    window_index_in_day        0.0676
    failed_conn_ratio          0.0580
    ttl_mean                   0.0481
    fwd_bwd_ratio              0.0457
    flow_count                 0.0400
    tcp_ratio                  0.0372
    syn_rate                   0.0326
[explain] -> C:\Users\chall\OneDrive\Desktop\SIH\artifacts\reports\explainability.json


---
All artefacts under `../../runs/` (benchmarks, models, figures, simulations,
explainability) and `../../artifacts/`. CLI equivalent: `python -m sentinel_wm.research all`.